# BacNav Sequence Filtering

This notebook filters raw NCBI BacNav search results based on specific criteria. It takes a FASTA file and performs the following steps:

## Filtering Criteria

- Removes sequences **shorter than 100 amino acids**
- Removes sequences **longer than 1000 amino acids**
- Keeps only sequences that contain the descriptive terms:
  - `"voltage"`
  - `"sodium"`

> ✅ This ensures we only keep **Voltage-Gated Sodium Channels**.

## Sequence Clustering with CD-HIT

- **First Pass:** Uses **CD-HIT** to eliminate sequences with **≥90% similarity**, keeping only **unique** sequences.
- **Second Pass:** Runs **CD-HIT** again to **cluster** the unique sequences.

---

_Note: Make sure `cd-hit`,  `biopython`,  `tdqm` are installed and accessible from your environment._


## Filtering by Sequence Length

This part of the code processes the raw FASTA file and filters out sequences based on their length:

- **Removes sequences shorter than 100 amino acids**
- **Removes sequences longer than 500 amino acids**


In [3]:
from Bio import SeqIO
import os

# === INPUT FILE NAME ===
input_file = "data_sources/vgsc-raw.fasta"  # Change this if your file has a different name
output_file = "results/sequences/raw_min100_max500.fasta"

# === FILTER SEQUENCES ===
min_len = 100
max_len = 500

# Read and filter sequences
filtered_seqs = []
removed_count = 0

for record in SeqIO.parse(input_file, "fasta"):
    seq_len = len(record.seq)
    if min_len <= seq_len <= max_len:
        filtered_seqs.append(record)
    else:
        removed_count += 1

# Save the filtered sequences to the new file
SeqIO.write(filtered_seqs, output_file, "fasta")

# === PRINT STATS ===
print(f"Total sequences removed: {removed_count}")
print(f"Filtered sequences saved to: {output_file}")


Total sequences removed: 23
Filtered sequences saved to: results/sequences/raw_min100_max500.fasta


## Keyword-Based Filtering

This part of the code filters the sequences based on descriptive terms found in their headers or annotations. It **keeps only** sequences that contain one or both of the following keywords:

- `"voltage"`
- `"sodium"`

✅ This ensures that only **Voltage-Gated Sodium Channels** are retained for further analysis.


In [7]:
from Bio import SeqIO
import os

# === INPUT FILE NAME ===
input_file = "results/sequences/raw_min100_max500.fasta"  # Change this to your input file
output_file = "results/sequences/vgsc_min100_max500.fasta"

# === FILTER CRITERIA ===
min_len = 100
max_len = 500
keywords = ["voltage", "sodium"]

# Read and filter sequences
filtered_seqs = []
removed_count = 0

for record in SeqIO.parse(input_file, "fasta"):
    seq_len = len(record.seq)
    desc = record.description.lower()
    
    # Check length and keywords in description
    if min_len <= seq_len <= max_len and all(keyword in desc for keyword in keywords):
        filtered_seqs.append(record)
    else:
        removed_count += 1

# Save filtered sequences
SeqIO.write(filtered_seqs, output_file, "fasta")

# === PRINT STATS ===
print(f"Total sequences removed: {removed_count}")
print(f"Filtered sequences saved to: {output_file}")


Total sequences removed: 265
Filtered sequences saved to: results/sequences/vgsc_min100_max500.fasta


## ✅ Sequence Clustering with CD-HIT (Part 1)

CD-HIT was executed outside of this notebook environment using a dedicated **Ubuntu Linux virtual machine** to ensure optimal performance and compatibility.

#### Clustering Process – First Pass (90% Similarity Threshold)

- **Input:** 1,392 sequences  
- **Similarity threshold:** 90%  
- **Output:** 828 unique sequences  
- ✅ **564 sequences were eliminated** due to high similarity



## Taxonomy Assignment of 828 Sequences

This code uses the **NCBI taxonomy database** to obtain the lineage of the 828 unique sequences from the first clustering step. 

The taxonomic information helps classify sequences for further analysis.

> 🔍 NCBI taxonomy provides standardized lineage data for accurate classification.


In [33]:
import os
import csv
import time
import re
from Bio import Entrez, SeqIO
from tqdm import tqdm

# Set your email (REQUIRED by NCBI)
Entrez.email = "your-email@example.com"  # ← Replace with your actual email

# === INPUT FILE CONFIGURATION ===
fasta_file = "results/sequences/vgsc_min100_max500_cdhit_0.90.fasta"  # ← Your FASTA file
limit_to_first_n = None  # Limit to the first 10 proteins

# Prepare output paths based on input FASTA file
base_name = os.path.splitext(os.path.basename(fasta_file))[0]
output_dir = os.path.dirname(fasta_file)
output_file_csv = os.path.join(output_dir, f"{base_name}_taxonomy_lineages.csv")
output_file_missing = os.path.join(output_dir, f"{base_name}_missing_proteins.csv")

def safe_entrez_fetch(db, id, rettype=None, retmode="text", attempts=3, sleep=1):
    for attempt in range(attempts):
        try:
            handle = Entrez.efetch(db=db, id=id, rettype=rettype, retmode=retmode)
            result = handle.read() if retmode == "text" else Entrez.read(handle)
            handle.close()
            return result
        except Exception as e:
            print(f"⚠️ Retry {attempt + 1}/{attempts} for ID {id} in {db}: {e}")
            time.sleep(sleep)
    print(f"❌ Failed to fetch {id} from {db} after {attempts} attempts.")
    return None

def get_organism_and_taxonomy(protein_id):
    try:
        time.sleep(0.34)  # Rate limit
        record = safe_entrez_fetch("protein", protein_id, rettype="gb", retmode="text")
        if record is None:
            return "Organism not found", None

        organism = "Organism not found"
        taxonomy_id = None
        for line in record.splitlines():
            if line.startswith("  ORGANISM"):
                organism = line.split("  ORGANISM")[1].strip()
            elif 'db_xref="taxon:' in line:
                taxonomy_id = line.split('taxon:')[1].split('"')[0]

        return organism, taxonomy_id
    except Exception as e:
        print(f"❌ Error fetching organism/taxonomy for protein {protein_id}: {e}")
        return "Organism not found", None

def get_lineage_from_taxonomy_id(taxonomy_id):
    try:
        if not taxonomy_id:
            return []

        time.sleep(0.34)
        record = safe_entrez_fetch("taxonomy", taxonomy_id, retmode="xml")
        if not record:
            return []

        lineage_data = record[0].get("Lineage", "")
        lineage_list = [level.strip() for level in lineage_data.split(";") if level.strip()]
        lineage_list = [l for l in lineage_list if "group" not in l.lower() and "cluster" not in l.lower()]
        lineage_list = [l for l in lineage_list if l != "Oscillatoriophycideae"]

        return lineage_list
    except Exception as e:
        print(f"❌ Error fetching lineage for taxonomy ID {taxonomy_id}: {e}")
        return []

def process_fasta(fasta_file, output_file_csv, output_file_missing, limit=10):
    # Parse sequences into dict: protein_id -> sequence
    seq_dict = {record.id: str(record.seq) for record in SeqIO.parse(fasta_file, "fasta")}

    proteins = list(seq_dict.keys())
    if limit:
        proteins = proteins[:limit]

    cache = {}
    rows = []
    missing_proteins = []
    all_lineage_depths = set()

    for protein_id in tqdm(proteins, desc="🔍 Fetching taxonomy", unit="protein"):
        if protein_id in cache:
            organism, taxonomy_id, lineage_list = cache[protein_id]
        else:
            organism, taxonomy_id = get_organism_and_taxonomy(protein_id)
            lineage_list = get_lineage_from_taxonomy_id(taxonomy_id)
            cache[protein_id] = (organism, taxonomy_id, lineage_list)

        if organism == "Organism not found" and not lineage_list:
            print(f"❌ Protein not found: {protein_id}")
            missing_proteins.append(protein_id)

        all_lineage_depths.add(len(lineage_list))

        row = {
            "protein_id": protein_id,
            "taxonomy_id": taxonomy_id or "",
            "organism_from_ncb": organism,
            "_lineage_list": lineage_list,
            "sequence": seq_dict.get(protein_id, "")
        }
        rows.append(row)

    max_depth = max(all_lineage_depths) if all_lineage_depths else 0

    standard_names = {
        1: "cellular_root",
        2: "domain",
        3: "kingdom",
        4: "phylum",
        5: "class",
        6: "order",
        7: "family",
        8: "genus",
        9: "species",
        10: "subspecies"
    }

    lineage_columns = []
    for i in range(max_depth, 10, -1):
        lineage_columns.append(f"lineage_level_{i}")
    for i in range(10, 0, -1):
        lineage_columns.append(standard_names[i])

    final_rows = []
    for row in rows:
        lineage_list = row.pop("_lineage_list", [])
        padded_lineage = lineage_list + [""] * (max_depth - len(lineage_list))

        for i in range(max_depth):
            level_idx = i + 1
            col_name = standard_names.get(level_idx, f"lineage_level_{level_idx}")
            row[col_name] = padded_lineage[i]

        # Fix species if missing or empty by parsing organism name
        if not row.get("species") or row.get("species") == "":
            org_name = row.get("organism_from_ncb", "")
            parts = org_name.split()
            if len(parts) >= 2:
                row["species"] = parts[0] + " " + parts[1]
            elif row.get("genus"):
                row["species"] = row["genus"] + " sp."

        # Extract subspecies if possible
        if not row.get("subspecies") or row.get("subspecies") == "":
            if "subsp." in row["organism_from_ncb"]:
                match = re.search(r"subsp\.\s+([^\s]+)", row["organism_from_ncb"])
                if match:
                    row["subspecies"] = match.group(1)
            else:
                row["subspecies"] = ""

        final_rows.append(row)

    fieldnames = ["protein_id", "taxonomy_id", "organism_from_ncb"] + lineage_columns + ["sequence"]

    with open(output_file_csv, mode='w', newline='') as out_file:
        writer = csv.DictWriter(out_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(final_rows)

    with open(output_file_missing, mode='w', newline='') as miss_file:
        writer = csv.writer(miss_file)
        writer.writerow(["protein_id"])
        for pid in missing_proteins:
            writer.writerow([pid])

    print(f"\n✅ Output written to:\n  {output_file_csv}\n  {output_file_missing}")

# === Run the script ===
process_fasta(fasta_file, output_file_csv, output_file_missing, limit=limit_to_first_n)


🔍 Fetching taxonomy: 100%|█████████████████████████████████████████████████████| 828/828 [26:21<00:00,  1.91s/protein]


✅ Output written to:
  results/sequences\vgsc_min100_max500_cdhit_0.90_taxonomy_lineages.csv
  results/sequences\vgsc_min100_max500_cdhit_0.90_missing_proteins.csv


### ✅ CD-HIT Re-Clustering (Part 2)

Following the first pass, the 825 unique sequences were clustered again using CD-HIT to group sequences at a broader similarity range.

#### Clustering Process – Second Pass (70% Similarity Threshold)

- **Input:** 825 unique sequences  
- **Similarity threshold:** 70%  
- **Output:** 342 clusters

> 🔁 This second clustering step helps to further reduce redundancy and identify broader representative groups for downstream analysis.